In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
mnist_train = pd.read_csv("../input/digit-recognizer/train.csv")
mnist_test = pd.read_csv("../input/digit-recognizer/test.csv")

In [ ]:
y_train1 = pd.get_dummies(mnist_train.label)
print(y_train1.shape)

In [ ]:
npx = mnist_train.drop(['label'], axis=1)
x_train1 = np.array(npx).reshape(42000,28,28,1)/255
print(x_train1.shape)

In [ ]:
from sklearn.model_selection import train_test_split #to split our dataset
x_train, x_val, y_train, y_val = train_test_split(x_train1, y_train1, test_size=0.1, random_state=101)

In [ ]:
npx = mnist_test
x_test = np.array(npx).reshape(28000,28,28,1)/255
print(x_test.shape)

In [ ]:
import keras
from keras.models import Sequential
from keras.optimizers import adam_v2
from keras.utils.np_utils import to_categorical
from keras.layers import Dense, Dropout, Flatten, BatchNormalization
from keras.layers.convolutional import Conv2D, MaxPooling2D
from keras.preprocessing import image
from keras.callbacks import ReduceLROnPlateau

import numpy as np
import matplotlib.pyplot as plt
import cv2 
from PIL import Image 

np.random.seed(99)

In [ ]:
def display(img):
  plt.figure(figsize=(5,5))
  plt.axis("off")
  plt.imshow(img,cmap="gray")

display(x_train[1191])

In [ ]:
from keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(rotation_range = 10, width_shift_range = 0.1, shear_range = 0.1,
                             height_shift_range = 0.1, zoom_range = [0.9,1.1], fill_mode = 'nearest', 
                             horizontal_flip = False, vertical_flip = False)
datagen.fit(x_train)


In [ ]:
# PERUBAHAN DI BAGIAN INI
def Conv_Net_model():
  model = Sequential()
  
  # Blok Konvolusi 1
  model.add(Conv2D(32, (3,3), input_shape=(28,28,1), padding="same", activation="relu"))
  model.add(BatchNormalization())
  model.add(Conv2D(32, (3,3), padding="same", activation="relu")) # Tambahan Conv2D
  model.add(BatchNormalization())
  model.add(MaxPooling2D(pool_size=(2,2)))
  model.add(Dropout(0.25)) # Tambahan Dropout setelah pooling

  # Blok Konvolusi 2
  model.add(Conv2D(64, (3,3), activation="relu", padding="same"))
  model.add(BatchNormalization())
  model.add(Conv2D(64, (3,3), activation="relu", padding="same")) # Tambahan Conv2D
  model.add(BatchNormalization())
  model.add(MaxPooling2D(pool_size=(2,2)))
  model.add(Dropout(0.25)) # Tambahan Dropout setelah pooling

  # Lapisan Fully Connected (Dense)
  model.add(Flatten())
  model.add(Dense(256, activation="relu")) # Meningkatkan jumlah neuron dari 200
  model.add(BatchNormalization())
  model.add(Dropout(rate=0.5)) # Meningkatkan dropout rate
  
  # Lapisan Output
  model.add(Dense(10, activation="softmax"))
  
  return model

model = Conv_Net_model()
print(model.summary())

In [ ]:
#DIBAGIAN INI DIKASIH EARLY STOP
from keras.callbacks import EarlyStopping 

reduce_lr = ReduceLROnPlateau(monitor = 'val_accuracy', 
                              min_lr = 0.00001, verbose = 1,  patience = 2, factor = 0.7, mode = 'auto', cooldown = 0)

early_stop = EarlyStopping(monitor='val_accuracy', 
                           patience=5, 
                           restore_best_weights=True, 
                           verbose=1)

In [ ]:
#PERUBAHAN BATCH SIZE, EPOCHS DAN NAMBAH CALLBACK EARLYSTOP
opt = adam_v2.Adam(0.001, beta_1=0.9, beta_2=0.999, amsgrad=True)
model.compile(optimizer=opt, loss = "categorical_crossentropy", metrics = ["accuracy"])
h = model.fit(datagen.flow(x_train,y_train, batch_size=64), epochs=50, 
              validation_data=(x_val,y_val), shuffle=1, verbose=1, callbacks=[reduce_lr, early_stop])

In [ ]:
# SEL BARU: Latih ulang model pada keseluruhan data (train + validation)

print("--- Memulai Pelatihan Ulang pada Data Penuh ---")

# 1. Tentukan jumlah epoch optimal dari pelatihan sebelumnya
# Kita ambil jumlah epoch yang dijalankan sebelum EarlyStopping menghentikannya.
optimal_epochs = len(h.history['val_accuracy'])
print(f"Jumlah epoch optimal yang ditemukan: {optimal_epochs}")

# 2. Buat ulang dan kompilasi ulang model agar bobotnya di-reset dari awal
# Ini penting agar kita melatih model yang "fresh" pada data penuh.
model = Conv_Net_model()
opt = adam_v2.Adam(0.001, beta_1=0.9, beta_2=0.999, amsgrad=False)
model.compile(optimizer=opt, loss = "categorical_crossentropy", metrics = ["accuracy"])

# 3. Latih model pada 100% data latih (x_train1 dan y_train1)
# Perhatikan: Tidak ada validation_data dan tidak ada callbacks di sini.
# Kita sudah tahu resep terbaiknya, jadi kita hanya "memasak" saja.
model.fit(datagen.flow(x_train1, y_train1, batch_size=64),
          epochs=optimal_epochs, # Gunakan jumlah epoch optimal
          shuffle=1, 
          verbose=1)

print("--- Pelatihan Ulang Selesai ---")

In [ ]:
# SEL INI MENGGANTIKAN SEL PREDIKSI YANG LAMA DENGAN LOGIKA TTA
# y_pred=model.predict(x_test)
# y_test = np.argmax(y_pred,axis=1)
# print(y_test.shape)

#---------------------------------------------------------------------------------------------

# SEL INI MENGGANTIKAN SEL PREDIKSI YANG LAMA DENGAN LOGIKA TTA YANG BENAR
print("--- Memulai Prediksi dengan Test Time Augmentation (TTA) ---")

# Jumlah augmentasi yang akan kita buat untuk setiap gambar tes. 10 adalah angka yang baik.
tta_steps = 9 
all_predictions = []

# Langkah 1: Buat prediksi pada data tes asli (tanpa augmentasi)
print(f"Step 1/{tta_steps + 1}: Memprediksi gambar asli...")
# Gunakan batch_size yang lebih besar untuk prediksi agar lebih cepat
y_pred_original = model.predict(x_test, batch_size=128, verbose=1) 
all_predictions.append(y_pred_original)

# Langkah 2: Buat prediksi pada gambar tes yang di-augmentasi secara berulang
# Kita akan menggunakan 'datagen' yang sudah Anda definisikan di sel ke-9
for i in range(tta_steps):
    print(f"Step {i + 2}/{tta_steps + 1}: Memprediksi batch augmented ke-{i+1}...")
    # Buat generator baru untuk setiap iterasi TTA
    tta_datagen = datagen.flow(x_test, batch_size=128, shuffle=False)
    # Lakukan prediksi pada data augmented
    preds = model.predict(tta_datagen, verbose=1)
    all_predictions.append(preds)
    # Hentikan generator setelah satu epoch penuh
    if len(all_predictions) > tta_steps:
        break

# Langkah 3: Rata-ratakan semua prediksi yang telah dikumpulkan
print("\n--- Merata-ratakan semua hasil prediksi TTA ---")
# np.mean akan menghitung rata-rata dari semua array prediksi di sepanjang sumbu 0
avg_predictions = np.mean(all_predictions, axis=0)

# Langkah 4: Dapatkan label kelas akhir dari prediksi rata-rata
y_test = np.argmax(avg_predictions, axis=1)

print("--- Prediksi TTA Selesai ---")
print(f"Bentuk akhir y_test: {y_test.shape}")

In [ ]:
sample = pd.read_csv("../input/digit-recognizer/sample_submission.csv")
i = sample.ImageId

output = pd.DataFrame({'ImageId' : i,'Label': y_test})
output.head(20)

In [ ]:
display(x_test[18])

In [ ]:
output.to_csv('submission.csv', index=False)